In [ ]:
# Cell 1: Setup — install deps, prep torch, clone repos
import os, subprocess, shutil, torch

print("Token not required — cloning public repo.")

# Find system torch path (Kaggle ships with PyTorch)
TORCH_DIR = os.path.dirname(torch.__file__)
print(f"System torch: {TORCH_DIR}")

REPO = "https://github.com/vfxjamer/talonbot.git"
DATASET = "peroplayer/talon-v1-checkpoints"

!rm -rf /kaggle/working/talonbot
!git clone {REPO} /kaggle/working/talonbot
print("Project cloned.")

# Restore checkpoints from Kaggle dataset
ckpt_dir = "/kaggle/working/talonbot/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
ret = subprocess.run(
    ["kaggle", "datasets", "download", DATASET, "--path", ckpt_dir, "--unzip", "--force"],
    capture_output=True, text=True
)
if ret.returncode == 0:
    print(f"Checkpoints restored from Kaggle dataset {DATASET}")
else:
    print(f"No prior checkpoints on Kaggle ({ret.stderr.strip()})")

# Run setup with full output
ret = subprocess.run(["bash", "/kaggle/working/talonbot/setup.sh"], capture_output=True, text=True)
if ret.returncode != 0:
    print("=== FULL STDOUT ===")
    print(ret.stdout[-8000:] if len(ret.stdout) > 8000 else ret.stdout)
    print("=== STDERR ===")
    print(ret.stderr[-2000:] if len(ret.stderr) > 2000 else ret.stderr)
    raise RuntimeError(f"Setup failed (exit {ret.returncode})")
else:
    print(ret.stdout[-2000:] if len(ret.stdout) > 2000 else ret.stdout)
print("Setup done.")

In [ ]:
# Cell 2: Build TalonBot
import os, subprocess, torch

TORCH_CMAKE = os.path.join(os.path.dirname(torch.__file__), "share", "cmake", "Torch")
os.environ["GIGALEARN_ROOT"] = "/workspace/libs/GigaLearnCPP"

cmake_cmd = [
    "cmake", "..",
    "-DCMAKE_BUILD_TYPE=RelWithDebInfo",
    "-DCMAKE_POSITION_INDEPENDENT_CODE=ON",
    f"-DGIGALEARN_ROOT=/workspace/libs/GigaLearnCPP",
    f"-DTorch_DIR={TORCH_CMAKE}"
]

!cd /kaggle/working/talonbot && rm -rf build && mkdir build && cd build && \
  {' '.join(cmake_cmd)} 2>&1 && \
  cmake --build . --config Release --target TalonBot -j$(nproc) 2>&1
print("Build done.")

In [ ]:
# Cell 3: Run training
import subprocess, sys, os

CWD = "/kaggle/working/talonbot"
ckpt_dir = os.path.join(CWD, "checkpoints")

latest = 0
if os.path.exists(ckpt_dir):
    for d in os.listdir(ckpt_dir):
        p = os.path.join(ckpt_dir, d)
        if os.path.isdir(p) and d.isdigit():
            latest = max(latest, int(d))

phase = 1 if latest < 30_000_000_000 else 2 if latest < 100_000_000_000 else 3 if latest < 220_000_000_000 else 4 if latest < 340_000_000_000 else 5

print(f"{'='*50}")
print(f"  Phase {phase}  |  Checkpoint: {latest:,}")
print(f"{'='*50}")

p = subprocess.Popen(
    [os.path.join(CWD, "build", "TalonBot"), "/workspace/libs/collision_meshes"],
    cwd=CWD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True
)
for line in p.stdout:
    print(line, end=""); sys.stdout.flush()
p.wait()
print(f"\nExit code: {p.returncode}")

In [ ]:
# Cell 4: Backup daemon — push checkpoints to Kaggle dataset
import subprocess, os, json, time

DATASET = "peroplayer/talon-v1-checkpoints"
CKPT_DIR = "/kaggle/working/talonbot/checkpoints"

# Ensure dataset-metadata.json exists
meta = {
    "title": "Talos V1 Checkpoints",
    "id": DATASET,
    "licenses": [{"name": "CC0-1.0"}]
}
with open(os.path.join(CKPT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f)

def backup_loop():
    while True:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", CKPT_DIR, "-m", "checkpoint", "--dir-mode", "zip"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"[BACKUP] Pushed at {time.strftime('%H:%M:%S UTC', time.gmtime())}")
        else:
            err = result.stderr.strip()
            if "404" in err or "Not Found" in err:
                # Dataset doesn't exist yet — create it
                subprocess.run(
                    ["kaggle", "datasets", "create", "-p", CKPT_DIR, "--public"],
                    capture_output=True
                )
                print(f"[BACKUP] Created dataset {DATASET}")
            else:
                print(f"[BACKUP] {err}")
        time.sleep(3600)

import threading
t = threading.Thread(target=backup_loop, daemon=True)
t.start()
print(f"[BACKUP] Daemon started — pushes {DATASET} every 60min")